# ĐỒ ÁN TOÁN ỨNG DỤNG VÀ THỐNG KÊ
# PHÉP KHỬ GAUSS VÀ CÁC ỨNG DỤNG

- **Nhóm thực hiện:** Nhóm 14 - Lớp 24CTT2
- **Mục tiêu:** Cài đặt phép khử Gauss có partial pivoting từ đầu bằng Python để giải hệ phương trình tuyến tính, tính định thức, tìm ma trận nghịch đảo, tính hạng và tìm cơ sở.

# Thiết lập cấu hình (config)
**Khai báo thư viện, hằng số EPSILON, Import các hàm thuật toán và thêm các hàm bổ trợ**

In [12]:
import sys, os
import pandas as pd
from IPython.display import display, Markdown

current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
if current_dir not in sys.path:
    sys.path.append(current_dir)
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

from config import EPSILON, is_zero, make_zero, AutoTestReporter

from gaussian import (gaussian_eliminate,verify_test_back_substitution, verify_test_gaussian_eliminate)
from determinant import (determinant, verify_test_determinant)
from inverse import (inverse, verify_test_inverse)
from rank_basis import (rank_and_basis, verify_test_rank_and_basis)
from test_case import (
    BACK_SUBSTITUTION_TEST_CASES,
    DETERMINANT_TEST_CASES,
    GAUSSIAN_ELIMINATE_TEST_CASES,
    INVERSE_TEST_CASES,
    RANK_BASIS_TEST_CASES,
    VERIFY_SOLUTION_TEST_CASES,
)

from verification import (verify_solution, verify_test_verify_solution,
    verify_determinant_numpy,  
    verify_inverse_numpy, 
    verify_rank_and_basis_numpy)


def display_matrix(matrix, title="Matrix", cmap="coolwarm"):
    """
    Sử dụng heatmap và pandas để hiển thị ma trận
    """
    print(f"\n--- {title} ---")
    
    if matrix is None or not matrix:
        print("   (Không tồn tại)")
        return
        
    df = pd.DataFrame(matrix)
    
    # 3. Xử lý riêng cho Vector 1 chiều: Ép thành 1 hàng ngang cho gọn
    if isinstance(matrix, list) and not isinstance(matrix[0], list):
        df = pd.DataFrame(matrix)
    
    # 4. Khử sai số float: Đưa các số cực nhỏ (gần 0) về hẳn 0.0 để bảng sạch sẽ
    df = df.map(lambda x: 0.0 if abs(x) < EPSILON else x) 
    
    styled_df = df.style.background_gradient(cmap=cmap, axis=None) \
                        .format("{:.4f}") \
                        .set_properties(**{
                            'text-align': 'center', 
                            'border': '1px solid #bbbbbb',
                            'padding': '10px'
                        })
    
    display(styled_df)
    
print(f"EPSILON có giá trị là: {EPSILON}")

EPSILON có giá trị là: 1e-15


# Phần 1: Demo các thuật toán

**1. Giải hệ phương trình**

In [13]:

display(Markdown("## HỆ CÓ NGHIỆM DUY NHẤT"))
A1 = [[1, 2, 3], 
     [0, 1, 4], 
     [5, 6, 0]]
b1 = [14, 32, 50]

display_matrix(A1, "Ma trận hệ số A")
display_matrix(b1, "Vector vế phải b")
Ab1_init = [row + [b1[i]] for i, row in enumerate(A1)]
display_matrix(Ab1_init, "Ma trận ghép [A|b] BAN ĐẦU")

# Thực thi thuật toán
M_res1, sol1, swaps1 = gaussian_eliminate(A1, b1)
display_matrix(M_res1, "Ma trận bậc thang [A|b] sau khi khử Gauss")
print(f"Số lần hoán đổi dòng: {swaps1}")
display_matrix(sol1, "Nghiệm của hệ x", cmap="Greens")

print(">> KIỂM CHỨNG:")
is_correct1 = verify_solution(A1, b1, sol1)
print(f"Kết quả đối chiếu NumPy: {'ĐÚNG' if is_correct1 else 'SAI'}")


display(Markdown("## HỆ VÔ SỐ NGHIỆM"))
A2 = [[1, 2, 3], 
         [2, 4, 6], 
         [3, 6, 9]]
b2 = [2, 4, 6]

display_matrix(A2, "Ma trận hệ số A")
display_matrix(b2, "Vector vế phải b")
Ab2_init = [row + [b2[i]] for i, row in enumerate(A2)]
display_matrix(Ab2_init, "Ma trận ghép [A|b] BAN ĐẦU")

M_res2, sol2, swaps2 = gaussian_eliminate(A2, b2)
display_matrix(M_res2, "Ma trận bậc thang [A|b] sau khi khử Gauss")
print(f"Số lần hoán đổi dòng: {swaps2}")
print(f"Kết quả: {sol2}")

print("\n>> KIỂM CHỨNG:")
is_correct2 = verify_solution(A2, b2, sol2)
print(f"Kết quả đối chiếu NumPy: {'ĐÚNG' if is_correct2 else 'SAI'}")



display(Markdown("## HỆ VÔ NGHIỆM"))
A3 = [[1, 2, 3], 
         [2, 4, 6], 
         [3, 6, 9]]
b3 = [2, 4, 100] 
display_matrix(A3, "Ma trận hệ số A")
display_matrix(b3, "Vector vế phải b")
Ab3_init = [row + [b3[i]] for i, row in enumerate(A3)]
display_matrix(Ab3_init, "Ma trận ghép [A|b] BAN ĐẦU")

sol3 = None
try:
    M_res3, sol3, swaps3 = gaussian_eliminate(A3, b3)
    display_matrix(M_res3, "Ma trận sau khi khử Gauss")
except ValueError as e:
    print(f"THÔNG BÁO: {e.args[0]}")
    if len(e.args) > 1:
        M_error = e.args[1]
        display_matrix(M_error, "Ma trận [A|b] sau khi khử Gauss", cmap="Reds")

print("\n>> KIỂM CHỨNG:")
is_correct3 = verify_solution(A3, b3, sol3) 
print(f"Kết quả đối chiếu NumPy: {'ĐÚNG' if is_correct3 else 'SAI'}")

## HỆ CÓ NGHIỆM DUY NHẤT


--- Ma trận hệ số A ---


,0,1,2
0,1.0000,2.0000,3.0000
1,0.0000,1.0000,4.0000
2,5.0000,6.0000,0.0000



--- Vector vế phải b ---


,0
0,14.0000
1,32.0000
2,50.0000


...


--- Vector vế phải b ---


,0
0,2.0000
1,4.0000
2,100.0000



--- Ma trận ghép [A|b] BAN ĐẦU ---


,0,1,2,3
0,1.0000,2.0000,3.0000,2.0000
1,2.0000,4.0000,6.0000,4.0000
2,3.0000,6.0000,9.0000,100.0000


THÔNG BÁO: Hệ phương trình vô nghiệm.

>> KIỂM CHỨNG:
Kết quả đối chiếu NumPy: ĐÚNG


**2. Tính định thức ma trận**

In [14]:
A_det = [[1, 2, 3], 
         [0, 1, 4], 
         [5, 6, 0]]

det_val = determinant(A_det)

display_matrix(A_det, "Ma trận cần tính định thức")
print(f"\n>> Giá trị định thức: {det_val:.4f}")

is_ok_det = verify_determinant_numpy(A_det, det_val)
print(f">> Kiểm chứng NumPy: {'ĐÚNG' if is_ok_det else 'SAI'}")


--- Ma trận cần tính định thức ---


,0,1,2
0,1.0000,2.0000,3.0000
1,0.0000,1.0000,4.0000
2,5.0000,6.0000,0.0000



>> Giá trị định thức: 1.0000
>> Kiểm chứng NumPy: ĐÚNG


**3. Tìm ma trận nghịch đảo** 

In [15]:
A_inv = [[1, 2, 3], 
              [0, 1, 4], 
              [5, 6, 0]]

inv_res = inverse(A_inv)

display_matrix(A_inv, "Ma trận gốc")
display_matrix(inv_res, "Ma trận nghịch đảo A^-1", cmap="coolwarm")

is_ok_inv = verify_inverse_numpy(A_inv, inv_res)
print(f">> Kiểm chứng NumPy: {'ĐÚNG' if is_ok_inv else 'SAI'}")


--- Ma trận gốc ---


,0,1,2
0,1.0000,2.0000,3.0000
1,0.0000,1.0000,4.0000
2,5.0000,6.0000,0.0000



--- Ma trận nghịch đảo A^-1 ---


,0,1,2
0,-24.0000,18.0000,5.0000
1,20.0000,-15.0000,-4.0000
2,-5.0000,4.0000,1.0000


>> Kiểm chứng NumPy: ĐÚNG


**4. Hạng và cơ sở của ma trận** 

In [16]:

A= [[1, 2, 1, 1],
    [2, 4, 2, 2],
    [3, 6, 3, 3]]

display_matrix(A, "Ma trận A")

rank,r_basis, c_basis, n_basis  = rank_and_basis(A)

print(f"\n>> Hạng của ma trận: {rank}")
display_matrix(r_basis, "Cơ sở không gian dòng ")
display_matrix(c_basis, "Cơ sở không gian cột", cmap="YlGnBu")
if n_basis:
    display_matrix(n_basis, "Cơ sở không gian nghiệm", cmap="Purples")
else:
    print("\n--- Cơ sở không gian nghiệm ---")
    print("   (Hệ chỉ có nghiệm tầm thường, Null Space = {0})")

results = verify_rank_and_basis_numpy(A, rank, r_basis, c_basis, n_basis)
is_all_correct = all(results)
print(f"\n=> Kết quả kiểm chứng tổng thể: {'ĐÚNG' if is_all_correct else 'SAI'}")


--- Ma trận A ---


,0,1,2,3
0,1.0000,2.0000,1.0000,1.0000
1,2.0000,4.0000,2.0000,2.0000
2,3.0000,6.0000,3.0000,3.0000



>> Hạng của ma trận: 1

--- Cơ sở không gian dòng  ---


,0,1,2,3
0,1.0000,2.0000,1.0000,1.0000



--- Cơ sở không gian cột ---


,0,1,2
0,1.0000,2.0000,3.0000



--- Cơ sở không gian nghiệm ---


,0,1,2,3
0,-2.0000,1.0000,0.0000,0.0000
1,-1.0000,0.0000,1.0000,0.0000
2,-1.0000,0.0000,0.0000,1.0000



=> Kết quả kiểm chứng tổng thể: ĐÚNG


# Phần 2: Kiểm tra Test Case

**1. Test - Back Substitution** 

In [17]:
display(Markdown("## BACK SUBSTITUTION"))

display(Markdown("### Danh sách các nội dung cần test"))
df_back_substitution_tests = pd.DataFrame(BACK_SUBSTITUTION_TEST_CASES)
columns_to_show = ['Nội dung', 'Ma trận U', 'Vector cột c'] 
valid_columns = [col for col in columns_to_show if col in df_back_substitution_tests.columns]

display(
    df_back_substitution_tests[valid_columns].style.set_properties(
        **{'text-align': 'left', 'border': '1px solid #ddd'}
    ).set_table_styles([
        {'selector': 'th', 'props': [('text-align', 'left'), ('border', '1px solid #ddd')]}
    ])
)

display(Markdown("### Tiến hàng tự động test:"))
verify_test_back_substitution(BACK_SUBSTITUTION_TEST_CASES)

## BACK SUBSTITUTION

### Danh sách các nội dung cần test

,Nội dung,Ma trận U,Vector cột c
0,Hệ 2x2,"[[2.0, 1.0], [0.0, 4.0]]","[5.0, 8.0]"
1,Hệ 3x3,"[[3.0, 0.0, 0.0], [0.0, -2.0, 0.0], [0.0, 0.0, 5.0]]","[9.0, 4.0, -10.0]"
2,Hệ 4x4 đơn giản,"[[1.0, 2.0, -1.0, 1.0], [0.0, -1.0, 3.0, 0.0], [0.0, 0.0, 2.0, -2.0], [0.0, 0.0, 0.0, 3.0]]","[2.0, 5.0, -2.0, 6.0]"
3,Có số 0 ở đường chéo chính,"[[1.0, 2.0, 3.0], [0.0, 0.0, 4.0], [0.0, 0.0, 5.0]]","[1.0, 2.0, 3.0]"
4,Ma trận hệ số có giá trị cực lớn,"[[100000000.0, -1.0], [0.0, 200000000.0]]","[99999998.0, 400000000.0]"
5,Hệ 1x1,[[5.0]],[10.0]


### Tiến hàng tự động test:

Hệ 2x2                                                                 [OK]  
Hệ 3x3                                                                 [OK]  
Hệ 4x4 đơn giản                                                        [OK]  
Có số 0 ở đường chéo chính                                             [OK]  
Ma trận hệ số có giá trị cực lớn                                       [OK]  
Hệ 1x1                                                                 [OK]  


### Kết luận: 6/6 (100%) hoàn thành


**2. Test - Gaussian Elimination** 

In [18]:
display(Markdown("## GAUSSIAN ELIMINATION"))
display(Markdown("### Danh sách các ma trận được sử dụng để kiểm thử:"))
df_gauss_tests = pd.DataFrame(GAUSSIAN_ELIMINATE_TEST_CASES)
columns_to_show = ['Nội dung', 'Ma trận A', 'Vector cột b'] 
valid_columns = [col for col in columns_to_show if col in df_gauss_tests.columns]

display(
    df_gauss_tests[valid_columns].style.set_properties(
        **{'text-align': 'left', 'border': '1px solid #ddd'}
    ).set_table_styles([
        {'selector': 'th', 'props': [('text-align', 'left'), ('border', '1px solid #ddd')]}
    ])
)

display(Markdown("### Tiến hành tự động test"))
verify_test_gaussian_eliminate(GAUSSIAN_ELIMINATE_TEST_CASES)

## GAUSSIAN ELIMINATION

### Danh sách các ma trận được sử dụng để kiểm thử:

,Nội dung,Ma trận A,Vector cột b
0,Hệ phương trình 2x2 nghiệm duy nhất,"[[2.0, 1.0], [1.0, -1.0]]","[4.0, -1.0]"
1,Bắt buộc Partial Pivoting (Phần tử a[0][0] = 0),"[[0.0, 2.0], [3.0, 1.0]]","[4.0, 5.0]"
2,Hệ 3x3 cơ bản,"[[2.0, 1.0, -1.0], [-3.0, -1.0, 2.0], [-2.0, 1.0, 2.0]]","[8.0, -11.0, -3.0]"
3,Hệ vô nghiệm,"[[1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [2.0, 2.0, 2.0]]","[3.0, 4.0, 5.0]"
4,Hệ vô số nghiệm,"[[1.0, -2.0, 1.0], [2.0, -4.0, 2.0], [3.0, -6.0, 3.0]]","[5.0, 10.0, 15.0]"
5,Cột đầu tiên là số cực nhỏ (Chống lỗi sai số chuẩn),"[[1e-15, 1.0], [1.0, 1.0]]","[2.0, 3.0]"
6,"Hệ nhiều phương trình hơn số ẩn, có nghiệm duy nhất","[[1.0, 1.0], [1.0, -1.0], [2.0, 1.0]]","[3.0, 1.0, 5.0]"


### Tiến hành tự động test

Hệ phương trình 2x2 nghiệm duy nhất                                    [OK]  
Bắt buộc Partial Pivoting (Phần tử a[0][0] = 0)                        [OK]  
Hệ 3x3 cơ bản                                                          [OK]  
Hệ vô nghiệm                                                           [OK]  
-> Bắt đúng lỗi: Hệ phương trình vô nghiệm.

Hệ có vô số nghiệm, công thức nghiệm tổng quát:
x = [5.0, 0.0, 0.0] + c1*[2.0, 1.0, 0.0] + c2*[-1.0, 0.0, 1.0]
Hệ vô số nghiệm                                                        [OK]  
Cột đầu tiên là số cực nhỏ (Chống lỗi sai số chuẩn)                    [OK]  
Hệ nhiều phương trình hơn số ẩn, có nghiệm duy nhất                    [OK]  


### Kết luận: 7/7 (100%) hoàn thành


**3. Test - Determinant** 

In [19]:
display(Markdown("## DETERMINANT"))


display(Markdown("### Danh sách các ma trận được sử dụng để kiểm thử:"))
df_determinant_tests = pd.DataFrame(DETERMINANT_TEST_CASES)
columns_to_show = ['Nội dung', 'Ma trận A'] 
valid_columns = [col for col in columns_to_show if col in df_determinant_tests.columns]

display(
    df_determinant_tests[valid_columns].style.set_properties(
        **{'text-align': 'left', 'border': '1px solid #ddd'}
    ).set_table_styles([
        {'selector': 'th', 'props': [('text-align', 'left'), ('border', '1px solid #ddd')]}
    ])
)

display(Markdown("### Tiến hành tự động test"))
verify_test_determinant(DETERMINANT_TEST_CASES)

## DETERMINANT

### Danh sách các ma trận được sử dụng để kiểm thử:

,Nội dung,Ma trận A
0,Ma trận 2x2,"[[1.0, 2.0], [3.0, 4.0]]"
1,Ma trận 3x3,"[[1.0, -9.4, -12.0], [2.0, -6.0, 5.0], [5.0, -7.0, 6.5]]"
2,Ma trận suy biến,"[[1.0, -2.0, 3.0], [-2.0, 4.0, -6.0], [5.0, 1.0, 2.0]]"
3,Ma trận không vuông,"[[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]]"
4,Ma trận 4x4,"[[1.0, 0.0, 2.0, -1.0], [3.0, 0.0, 0.0, 5.0], [2.0, 1.0, 4.0, -3.0], [1.0, 0.0, 5.0, 0.0]]"
5,Ma trận 1x1,[[1.0]]
6,Ma trận toàn số 0,"[[0.0, 0.0], [0.0, 0.0]]"


### Tiến hành tự động test

Ma trận 2x2                                                            [OK]  (det = -2.0000)
Ma trận 3x3                                                            [OK]  (det = -308.8000)
Ma trận suy biến                                                       [OK]  (det = 0.0000)
Ma trận không vuông                                                    [OK]  
-> Bắt đúng lỗi: ValueError:Ma trận không vuông, không thể tính định thức.
Ma trận 4x4                                                            [OK]  (det = 30.0000)
Ma trận 1x1                                                            [OK]  (det = 1.0000)
Ma trận toàn số 0                                                      [OK]  (det = 0.0000)


### Kết luận: 7/7 (100%) hoàn thành


**4. Test - Inverse** 

In [20]:
display(Markdown("## INVERSE"))
display(Markdown("### Danh sách các ma trận được sử dụng để kiểm thử: "))
df_inverse_tests = pd.DataFrame(INVERSE_TEST_CASES)
columns_to_show = ['Nội dung', 'Ma trận A'] 
valid_columns = [col for col in columns_to_show if col in df_inverse_tests.columns]

display(
    df_inverse_tests[valid_columns].style.set_properties(
        **{'text-align': 'left', 'border': '1px solid #ddd'}
    ).set_table_styles([
        {'selector': 'th', 'props': [('text-align', 'left'), ('border', '1px solid #ddd')]}
    ])
)

display(Markdown("### Tiến hành tự động test"))
verify_test_inverse(INVERSE_TEST_CASES)

## INVERSE

### Danh sách các ma trận được sử dụng để kiểm thử: 

,Nội dung,Ma trận A
0,Ma trận 2x2,"[[2.0, 5.0], [1.0, 3.0]]"
1,Ma trận 3x3 hoán vị,"[[0.0, 0.0, 1.0], [1.0, 0.0, 0.0], [0.0, 1.0, 0.0]]"
2,Ma trận suy biến,"[[4.0, 6.0], [2.0, 3.0]]"
3,Ma trận không vuông,"[[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]]"
4,Ma trận 1x1,[[4.0]]


### Tiến hành tự động test

Ma trận 2x2                                                            [OK]  
Ma trận 3x3 hoán vị                                                    [OK]  
Không có pivot tại cột 1
Ma trận suy biến                                                       [OK]  
Ma trận không vuông                                                    [OK]  
 -> Bắt đúng lỗi: Ma trận không vuông, không thể tìm nghịch đảo.
Ma trận 1x1                                                            [OK]  


### Kết luận: 5/5 (100%) hoàn thành


**5. Test - Rank and Basis** 

In [21]:
display(Markdown("## RANK AND BASIS"))
display(Markdown("### Danh sách các ma trận được sử dụng để kiểm thử: "))
df_rank_and_basis_tests = pd.DataFrame(RANK_BASIS_TEST_CASES)
columns_to_show = ['Nội dung', 'Ma trận A'] 
valid_columns = [col for col in columns_to_show if col in df_rank_and_basis_tests.columns]

display(
    df_rank_and_basis_tests[valid_columns].style.set_properties(
        **{'text-align': 'left', 'border': '1px solid #ddd'}
    ).set_table_styles([
        {'selector': 'th', 'props': [('text-align', 'left'), ('border', '1px solid #ddd')]}
    ])
)

display(Markdown("### Tiến hành tự động test"))
verify_test_rank_and_basis(RANK_BASIS_TEST_CASES)

## RANK AND BASIS

### Danh sách các ma trận được sử dụng để kiểm thử: 

,Nội dung,Ma trận A
0,Rank đầy đủ (Ma trận 3x3),"[[2.0, 0.0, -1.0], [4.0, -5.0, 2.0], [0.0, 0.0, 7.0]]"
1,Ma trận toàn số 0 (Rank = 0),"[[0.0, 0.0], [0.0, 0.0], [0.0, 0.0]]"
2,Ma trận chữ nhật 3x4 (Thực tế Rank = 2),"[[1.0, 2.0, 0.0, -1.0], [2.0, 6.0, -3.0, -3.0], [3.0, 10.0, -6.0, -5.0]]"
3,Các dòng phụ thuộc tuyến tính (Rank 1),"[[1.0, -3.0, 2.0], [-2.0, 6.0, -4.0], [3.0, -9.0, 6.0]]"
4,Ma trận chữ nhật dọc 4x2 (Rank tối đa = 2),"[[1.0, 2.0], [3.0, 4.0], [5.0, 6.0], [7.0, 8.0]]"
5,Cần hoán vị dòng (Pivot a[0][0] = 0),"[[0.0, 2.0, 1.0], [1.0, -1.0, 0.0], [0.0, 0.0, 3.0]]"
6,Chống sai số số học (Kiểm tra dung sai EPSILON),"[[1.0, 1.0], [1.0, 1.0]]"


### Tiến hành tự động test

Rank đầy đủ (Ma trận 3x3)                                              [OK]  
Ma trận toàn số 0 (Rank = 0)                                           [OK]  
Ma trận chữ nhật 3x4 (Thực tế Rank = 2)                                [OK]  
Các dòng phụ thuộc tuyến tính (Rank 1)                                 [OK]  
Ma trận chữ nhật dọc 4x2 (Rank tối đa = 2)                             [OK]  
Cần hoán vị dòng (Pivot a[0][0] = 0)                                   [OK]  
Chống sai số số học (Kiểm tra dung sai EPSILON)                        [OK]  


### Kết luận: 7/7 (100%) hoàn thành


**6. Test - Verify Solution** 

In [22]:
display(Markdown("## KIỂM CHỨNG KẾT QUẢ GAUSS"))
display(Markdown("### Danh sách các ma trận được sử dụng để kiểm thử: "))
df_verify_solution_tests = pd.DataFrame(VERIFY_SOLUTION_TEST_CASES)
columns_to_show = ['Nội dung', 'A', 'x', 'b'] 
valid_columns = [col for col in columns_to_show if col in df_verify_solution_tests.columns]

display(
    df_verify_solution_tests[valid_columns].style.set_properties(
        **{'text-align': 'left', 'border': '1px solid #ddd'}
    ).set_table_styles([
        {'selector': 'th', 'props': [('text-align', 'left'), ('border', '1px solid #ddd')]}
    ])
)

display(Markdown("### Tiến hành tự động test"))
verify_test_verify_solution(VERIFY_SOLUTION_TEST_CASES)

## KIỂM CHỨNG KẾT QUẢ GAUSS

### Danh sách các ma trận được sử dụng để kiểm thử: 

,Nội dung,A,x,b
0,Nghiệm trả về đúng,"[[3.0, 2.0], [1.0, -1.0]]","[1.0, 1.0]","[5.0, 0.0]"
1,Nghiệm trả về sai,"[[1.0, 2.0], [3.0, 4.0]]","[0.0, 0.0]","[5.0, 11.0]"
2,Ma trận Hilbert 4x4,"[[1.0, 0.5, 0.3333333333333333, 0.25], [0.5, 0.3333333333333333, 0.25, 0.2], [0.3333333333333333, 0.25, 0.2, 0.16666666666666666], [0.25, 0.2, 0.16666666666666666, 0.14285714285714285]]","[1.0, 1.0, 1.0, 1.0]","[2.0833333333333335, 1.2833333333333334, 0.95, 0.7595238095238095]"
3,Nghiệm chứa 0,"[[2.0, -1.0], [1.0, 1.0]]","[0.0, 3.0]","[-3.0, 3.0]"


### Tiến hành tự động test

Nghiệm trả về đúng                                                     [OK]  
Nghiệm trả về sai                                                      [OK]  
Ma trận Hilbert 4x4                                                    [OK]  
Nghiệm chứa 0                                                          [OK]  


### Kết luận: 4/4 (100%) hoàn thành
